<a href="https://colab.research.google.com/github/w902796-source/Agentic-AI-training-projects/blob/main/langchainpgm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Installation of Dependencies

First, let's install the required LangChain packages, the updated Google GenAI SDK, and dependencies for document processing and vector stores.

In [1]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.

### 2. Configure API Keys

To use Gemini models via LangChain, you need a Gemini API key. Ensure you have added your API key to Colab's Secrets manager (the key symbol on the left panel) with the name `GOOGLE_API_KEY`.

In [2]:
import os
from google.colab import userdata
from getpass import getpass

# Try loading from Secrets first, otherwise prompt the user
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("API Key loaded successfully from Secrets!")
except Exception:
    print("GOOGLE_API_KEY not found in Secrets. Please paste it below to continue:")
    api_key = getpass("Enter your Google API Key: ")
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key
        print("API Key set successfully!")
    else:
        print("No key provided. The following cells will fail without an API key.")

GOOGLE_API_KEY not found in Secrets. Please paste it below to continue:
Enter your Google API Key: ··········
API Key set successfully!


### 3. Load Documents and Split into Chunks

We will define some sample technical documentation (e.g., details about a fictional software API) and split it into manageable chunks for our vector database.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Sample technical documentation dataset
documents = [
    Document(
        page_content="""CloudWidget API Version 3.6 Overview:
CloudWidget is a cloud-native hosting service that deploys containers globally.
To deploy a new container, send a POST request to `/api/v2/deploy` with a JSON payload specifying `image_name` and `region`."""
    ),
    Document(
        page_content="""CloudWidget API Authentication:
All endpoints require an HTTP header `Authorization: Bearer <your_api_token>`.
Tokens can be generated from the CloudWidget Console under developer settings."""
    ),
    Document(
        page_content="""CloudWidget Pricing Tier:
- Free Tier: Includes 1 vCPU, 512MB RAM, and 10GB bandwidth per month.
- Pro Tier: Costs $15/month per container and includes 2 vCPUs, 2GB RAM, and 100GB bandwidth."""
    )
]

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
splits = text_splitter.split_documents(documents)
print(f"Split documents into {len(splits)} chunks.")

Split documents into 6 chunks.


### 4. Create Vector Store and Store Embeddings

We will generate embeddings using `GoogleGenerativeAIEmbeddings` and index them in a temporary FAISS vector database.

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
import os

# Retrieve the set API key safely
api_key = os.environ.get("GOOGLE_API_KEY", None)

# Initialize Google GenAI Embeddings using a supported model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2", # Changed to a model that supports embedContent
    google_api_key=api_key
)

# Populate vector store
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Vector store successfully initialized and populated!")

Vector store successfully initialized and populated!


In [8]:
import google.generativeai as genai
import os

# Ensure the API key is set for the genai library
api_key = os.environ.get("GOOGLE_API_KEY", None)
if api_key:
    genai.configure(api_key=api_key)
else:
    print("GOOGLE_API_KEY is not set. Please set it to list models.")

print("Available Generative Models:")
for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(f"  - {model.name} (Supports embedContent)")
    elif "generateContent" in model.supported_generation_methods:
        print(f"  - {model.name} (Supports generateContent)")
    else:
        print(f"  - {model.name} (Other methods)")

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Generative Models:
  - models/gemini-2.5-flash (Supports generateContent)
  - models/gemini-2.5-pro (Supports generateContent)
  - models/gemini-2.5-flash-preview-tts (Supports generateContent)
  - models/gemini-2.5-pro-preview-tts (Supports generateContent)
  - models/gemma-4-26b-a4b-it (Supports generateContent)
  - models/gemma-4-31b-it (Supports generateContent)
  - models/gemini-flash-latest (Supports generateContent)
  - models/gemini-flash-lite-latest (Supports generateContent)
  - models/gemini-pro-latest (Supports generateContent)
  - models/gemini-2.5-flash-lite (Supports generateContent)
  - models/gemini-2.5-flash-image (Supports generateContent)
  - models/gemini-3-flash-preview (Supports generateContent)
  - models/gemini-3.1-pro-preview (Supports generateContent)
  - models/gemini-3.1-pro-preview-customtools (Supports generateContent)
  - models/gemini-3.1-flash-lite-preview (Supports generateContent)
  - models/gemini-3.1-flash-lite (Supports generateContent)


### 5. Define the RAG Pipeline

We build a chain using LangChain Expression Language (LCEL) that retrieves relevant context chunks, passes them to a prompt template, and utilizes the `gemini-2.5-flash` model to formulate the answer.

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Retrieve the set API key safely
api_key = os.environ.get("GOOGLE_API_KEY", None)

# Define model explicitly passing the api_key
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key)

# Create prompt template
template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, say that you don't know.

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# Helper to format retrieved documents as a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain constructed successfully!")

RAG Chain constructed successfully!


### 6. Test the RAG Assistant

Let's query the assistant on topics related to the `CloudWidget` service.

In [14]:
query = "How do I authenticate requests to the CloudWidget API?"
response = rag_chain.invoke(query)

print(f"User Query: {query}\n")
print(f"Assistant Response:\n{response}")

User Query: How do I authenticate requests to the CloudWidget API?

Assistant Response:
To authenticate requests to the CloudWidget API, you need to include an HTTP header in all your endpoints using the following format:

`Authorization: Bearer <your_api_token>`

You can generate API tokens in the CloudWidget Console under developer settings.
